In [64]:
import os
from dotenv import load_dotenv
import easyocr
from pdf2image import convert_from_path
import numpy as np
import openai 
from openai import OpenAI
import json
from pythainlp.tokenize import word_tokenize
from pythainlp.spell import correct
from pythainlp.corpus.common import thai_words
from pythainlp.util import normalize
import re

In [65]:
client = OpenAI(
    api_key=os.getenv("OPENAI_API_KEY"),
    base_url="https://openrouter.ai/api/v1"
)

In [66]:
known_words = set(thai_words())
load_dotenv(override=True)
openai.api_key = os.getenv("OPENAI_API_KEY") 
assert openai.api_key, "❌ ไม่พบ OPENAI_API_KEY ใน .env"

In [67]:
POPPLER_PATH = r"C:/Users/Ned/Desktop/Poppler/poppler-24.08.0/Library/bin"

In [68]:
pdf_file = "bill.pdf"   # เปลี่ยนชื่อไฟล์ตามของคุณ
images = convert_from_path(
    pdf_file,
    dpi=300,
    poppler_path=POPPLER_PATH
)
print(f"✅ แปลง PDF เสร็จแล้ว ได้ {len(images)} หน้า")


✅ แปลง PDF เสร็จแล้ว ได้ 2 หน้า


In [69]:
# ✅ Custom whitelist: คำที่ห้ามแก้
custom_whitelist = {
    "วิทยาลัย", "ศิลปะ", "เทคโนโลยี", "บาท", "ไม่เกิน",
    "เฉพาะเจาะจง", "ใบขอซื้อ", "รายงาน", "อนุมัติ",
    "ผู้มีอำนาจ", "เลขที่", "ดร.", "น.ส.", "นาง", "นาย",
    "คุณ", "นางสาว", "รศ.", "ผศ.", "ศ.", "น.",
    "ปฏิสนธิ์", "ณัฐนิชา"
}

# ✅ คำที่รู้จักจาก PyThaiNLP corpus
known_words = set(thai_words())

# ✅ ล้าง noise เช่น .ณัฐนิชา → ณัฐนิชา
def clean_ocr_name(word: str) -> str:
    if word.startswith("."):
        return word[1:]
    return word

# ✅ ลบ prefix ที่เป็น noise เช่น ในดร. → ดร.
def remove_prefix_noise(word: str) -> str:
    if word.startswith("ในดร."):
        return word.replace("ในดร.", "ดร.")
    elif word.startswith("ใน"):
        return word[2:]
    return word

# ✅ ไม่แก้คำที่อยู่ใน whitelist หรือ known words
def safe_correct_word(word: str) -> str:
    if word in custom_whitelist or normalize(word) in known_words:
        return word
    return correct(word)

# ✅ ตรวจว่าเป็นวันที่แบบไทยหรือไม่ เช่น 31 มกราคม 2568
def is_probably_date(text: str) -> bool:
    thai_months = [
        "มกราคม", "กุมภาพันธ์", "มีนาคม", "เมษายน", "พฤษภาคม", "มิถุนายน",
        "กรกฎาคม", "สิงหาคม", "กันยายน", "ตุลาคม", "พฤศจิกายน", "ธันวาคม"
    ]
    for month in thai_months:
        if re.search(rf"\d{{1,2}}\s*{month}\s*\d{{4}}", text):
            return True
    return False

# ✅ ฟังก์ชันหลัก: แก้คำผิดอย่างปลอดภัย
def auto_correct_text(text: str):
    """
    คืนค่า (fixed_text, corrections) 
    - fixed_text: ข้อความที่แก้แล้ว
    - corrections: list ของ tuple (orig, fixed)
    """
    if is_probably_date(text):
        return text, []

    tokens = word_tokenize(text, engine="newmm")
    corrected_tokens = []
    corrections = []

    for w in tokens:
        w_clean = clean_ocr_name(w)
        w_fixed = safe_correct_word(w_clean)
        w_final = remove_prefix_noise(w_fixed)
        corrected_tokens.append(w_final)
        if w_final != w:
            corrections.append((w, w_final))

    return ''.join(corrected_tokens), corrections

# ✅ สร้าง EasyOCR Reader
reader = easyocr.Reader(['th', 'en'])

# ✅ กำหนดไฟล์สำหรับบันทึกผล
out_path = "output_corrected.txt"

# ✅ วน OCR ทุกหน้า แล้วบันทึกผล
with open(out_path, 'w', encoding='utf-8') as fout:
    fout.truncate(0)
    for pg, img in enumerate(images, start=1):
        fout.write(f"\n📄 Page {pg}\n")
        results = reader.readtext(np.array(img))
        for raw_bbox, raw_text, _ in results:
            fixed_text, corr = auto_correct_text(raw_text)
            fout.write(f"❌ {raw_text}\n")
            fout.write(f"✅ {fixed_text}\n")
            if corr:
                changes = ", ".join(f"{o}→{n}" for o, n in corr)
                fout.write(f"🔄 Corrections: {changes}\n")
            fout.write("\n")

print(f"✅ บันทึก OCR + แก้คำแล้วที่ '{out_path}'")

Neither CUDA nor MPS are available - defaulting to CPU. Note: This module is much faster with a GPU.


✅ บันทึก OCR + แก้คำแล้วที่ 'output_corrected.txt'


In [71]:
# อ่าน text ทั้งหมด
with open("output_corrected.txt", 'r', encoding='utf-8') as f:
    ocr_text = f.read()

def extract_fields(text: str) -> dict:
    system_prompt = (
    "คุณคือผู้ช่วย AI สำหรับดึงข้อมูลจากเอกสารทางการเงินของหน่วยงานราชการไทย "
    "ข้อความ OCR ที่คุณจะได้รับอาจมาจากตาราง หรือข้อความที่ไม่เรียงลำดับจากภาพจริง "
    "โปรดอ่านอย่างรอบคอบ และดึงข้อมูลที่สำคัญออกมาในรูปแบบ JSON เท่านั้น โดยมีฟิลด์ดังนี้:\n\n"
    "- bill_number: เลขที่ใบขอซื้อ เช่น 10778\n"
    "- bill_type: ประเภทเอกสาร เช่น รายงานขออนุมัติจัดจ้าง\n"
    "- supplier_name: หน่วยงานหรือชื่อผู้ขาย เช่น มหาวิทยาลัยเชียงใหม่\n"
    "- amount: ยอดรวมสุทธิที่อยู่ใกล้คำว่า 'รวมทั้งสิ้น' หรือ 'ยอดรวม'\n"
    "- payment_date: วันที่ใด ๆ ในเอกสาร เช่น 31 มกราคม 2568 หรือ 31 ม.ค. 2568\n"
    "- signature: รายชื่อผู้ลงนามทั้งหมด (เฉพาะชื่อจริงเท่านั้น)\n\n"

    "**ข้อควรระวังสำหรับ signature:**\n"
    "- ดึงเฉพาะชื่อบุคคล เช่น 'ณัฐนิชา วงศ์ปิน', 'ปฏิสนธิ ปาลี'\n"
    "- ห้ามใส่คำนำหน้า เช่น น.ส., ดร., ผศ. หรือ นาย\n"
    "- ห้ามใส่ตำแหน่ง เช่น รองคณบดี, หัวหน้าฝ่าย\n"
    "- ห้ามใส่วันที่ เช่น 31 ม.ค. 2568\n"
    "- หากไม่มีคำว่า 'ลงชื่อ' ให้ตรวจสอบชื่อที่อยู่ในรูปแบบ 'ชื่อ เว้นวรรค นามสกุล' แทน\n"
    "- ให้รวมชื่อทั้งหมดใน list เช่น [\"ณัฐนิชา วงศ์ปิน\", \"ลักษณ์ สมร่าง\"]\n\n"
    
    "คุณต้องตอบกลับเป็น JSON เท่านั้น หากข้อมูลใดไม่พบให้ใส่ null และห้ามเขียนคำอธิบายอื่นเพิ่มเติม"
    )
    user_prompt = f"ข้อมูล OCR ภาษาไทย:\n{text}\n\nกรุณาแสดงผลเป็น JSON เท่านั้น"

    response = client.chat.completions.create(
        model="openai/gpt-3.5-turbo",
        messages=[
            {"role": "system", "content": system_prompt},
            {"role": "user", "content": user_prompt}
        ],
        temperature=0
    )

    content = response.choices[0].message.content.strip()
    print("📥 AI response:\n", content)

    try:
        return json.loads(content)
    except json.JSONDecodeError as e:
        print("❌ JSON parsing failed:", e)
        return {"error": "Invalid JSON", "raw": content}

with open("output_corrected.txt", "r", encoding="utf-8") as f:
    ocr_text = f.read()

short_text = ocr_text[:1500]  

extracted = extract_fields(short_text)

print("📄 Extracted Fields:")
for key, value in extracted.items():
    print(f"• {key}: {value}")



📥 AI response:
 {
    "bill_number": "10778",
    "bill_type": "รายงานขอความเหบ่นขอบและขออนุมัติจัดซื้อเจใดจ้าง",
    "supplier_name": "มหาวิทยาลัยเชียงใหม",
    "amount": "วิธีเฉพาะเจาะจงไมเกิน1000บาท",
    "payment_date": "31 มกราคม 2568",
    "signature": null
}
📄 Extracted Fields:
• bill_number: 10778
• bill_type: รายงานขอความเหบ่นขอบและขออนุมัติจัดซื้อเจใดจ้าง
• supplier_name: มหาวิทยาลัยเชียงใหม
• amount: วิธีเฉพาะเจาะจงไมเกิน1000บาท
• payment_date: 31 มกราคม 2568
• signature: None
